In [ ]:
import os
import pandas as pd
import polars as pl
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')

import kaggle_evaluation.cmi_inference_server

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Set TensorFlow to be deterministic
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Configure GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

# Define gesture names
GESTURE_NAMES = [
    'Above ear - pull hair',           # 0 - BFRB
    'Cheek - pinch skin',              # 1 - BFRB
    'Drink from bottle/cup',           # 2 - NON-BFRB
    'Eyebrow - pull hair',             # 3 - BFRB
    'Eyelash - pull hair',             # 4 - BFRB
    'Feel around in tray and pull out an object',  # 5 - NON-BFRB
    'Forehead - pull hairline',        # 6 - BFRB
    'Forehead - scratch',              # 7 - BFRB
    'Glasses on/off',                  # 8 - NON-BFRB
    'Neck - pinch skin',               # 9 - BFRB
    'Neck - scratch',                  # 10 - BFRB
    'Pinch knee/leg skin',             # 11 - NON-BFRB
    'Pull air toward your face',       # 12 - NON-BFRB
    'Scratch knee/leg skin',           # 13 - BFRB
    'Text on phone',                   # 14 - NON-BFRB
    'Wave hello',                      # 15 - NON-BFRB
    'Write name in air',               # 16 - NON-BFRB
    'Write name on leg'                # 17 - NON-BFRB
]

gesture_classes = GESTURE_NAMES
BFRB_INDICES = {0, 1, 3, 4, 6, 7, 9, 10, 13}

class DataLoader:
    def __init__(self):
        self.train_df = None
        self.test_df = None
        self.train_demo = None
        self.test_demo = None

    def load_data(self):
        self.train_df = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/train.csv")
        self.train_demo = pd.read_csv("/kaggle/input/cmi-detect-behavior-with-sensor-data/train_demographics.csv")
        return self



class MissingDataHandler:
    def __init__(self):
        self.temp_cols = [f'thm_{i}' for i in range(1, 6)]
        self.tof_cols = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]

    def tof_failures(self, df):
        """Vectorized ToF handling"""
        tof_available = [col for col in self.tof_cols if col in df.columns]
        
        if tof_available:
            # Vectorized missing rate calculation per sensor
            for i in range(1, 6):
                sensor_cols = [col for col in tof_available if f'tof_{i}_' in col]
                if sensor_cols:
                    # Vectorized missing rate
                    sensor_data = df[sensor_cols]
                    df[f'tof_{i}_missing_rate'] = (sensor_data == -1).mean(axis=1)
                    
                    # Key pixels only - vectorized
                    key_pixels = [0, 7, 27, 28, 35, 36, 56, 63]
                    key_cols = [f'tof_{i}_v{p}' for p in key_pixels if f'tof_{i}_v{p}' in sensor_cols]
                    if key_cols:
                        df[f'tof_{i}_key_missing'] = (df[key_cols] == -1).mean(axis=1)

        return df

    def imputation(self, df, is_train=True):
        """Vectorized imputation"""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        exclude_cols = ['sequence_id', 'sequence_counter', 'row_id', 'subject', 'gesture']
        impute_cols = [col for col in numeric_cols if col not in exclude_cols and '_missing' not in col]

        if len(impute_cols) > 0:
            # Separate by sensor type
            imu_cols = [col for col in impute_cols if any(x in col for x in ['acc_', 'rot_'])]
            temp_cols = [col for col in impute_cols if 'thm_' in col]
            tof_cols = [col for col in impute_cols if 'tof_' in col and '_missing' not in col]
            
            # IMU imputation
            if imu_cols:
                medians = df[imu_cols].median()
                df[imu_cols] = df[imu_cols].fillna(medians)
            
            # Temperature imputation
            if temp_cols:
                medians = df[temp_cols].median()
                df[temp_cols] = df[temp_cols].fillna(medians)
            
            # ToF imputation
            if tof_cols:
                for col in tof_cols:
                    mask = df[col].isna()
                    if mask.any():
                        df.loc[mask, col] = 254  # 254 for true missing values
                
        return df

    def process(self, df, is_train=True):
        """Main processing function"""
        df = self.tof_failures(df)
        df = self.imputation(df, is_train=is_train)
        return df

    def imputation(self, df, is_train=True):
        """Vectorized imputation"""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        exclude_cols = ['sequence_id', 'sequence_counter', 'row_id', 'subject', 'gesture']
        impute_cols = [col for col in numeric_cols if col not in exclude_cols and '_missing' not in col]

        if len(impute_cols) > 0:
            # Vectorized fillna
            medians = df[impute_cols].median()
            df[impute_cols] = df[impute_cols].fillna(medians)
            
            # Special handling for ToF NaN
            tof_cols = [col for col in impute_cols if 'tof_' in col]
            if tof_cols:
                # Fill actual NaN, not -1
                for col in tof_cols:
                    mask = df[col].isna()
                    if mask.any():
                        df.loc[mask, col] = 254
                
        return df


class FeatureExtractor:
    def __init__(self):
        self.sensor_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z']
        self.temp_cols = [f'thm_{i}' for i in range(1, 6)]
        self.use_reduced_features = False

    def aggregate_sequence(self, df, is_train=True):
        """Fully vectorized sequence aggregation"""
        if 'sequence_id' not in df.columns:
            df['sequence_id'] = 1
        
        unique_seqs = df['sequence_id'].unique()
        
        if len(unique_seqs) == 1:
            return self._aggregate_single_sequence(df, unique_seqs[0], is_train)
        
        # Collecting all feature columns at once
        feature_cols = []
        
        # IMU features
        imu_cols = [col for col in self.sensor_cols if col in df.columns]
        feature_cols.extend(imu_cols)
        
        # Temperature features
        temp_available = [col for col in self.temp_cols if col in df.columns]
        if temp_available:
            df['temp_global_mean'] = df[temp_available].mean(axis=1)
            feature_cols.extend(temp_available + ['temp_global_mean'])
        
        # Missing indicators
        missing_cols = [col for col in df.columns if '_missing_rate' in col or '_key_missing' in col]
        feature_cols.extend(missing_cols)
        
        # Single aggregation for all features at once
        if feature_cols:
            agg_result = df.groupby('sequence_id')[feature_cols].agg(['mean', 'std', 'max', 'min'])
            
            # Flattening multi-level columns
            agg_result.columns = ['_'.join(col).strip() for col in agg_result.columns.values]
            agg_features = agg_result.reset_index()
            
            # Vectorized range calculation for acceleration
            acc_cols = [col for col in ['acc_x', 'acc_y', 'acc_z'] if col in df.columns]
            if acc_cols:
                # Using peak-to-peak for range
                ranges = df.groupby('sequence_id')[acc_cols].agg(np.ptp)
                ranges.columns = [f'{col}_range' for col in ranges.columns]
                agg_features = pd.concat([agg_features, ranges.reset_index(drop=True)], axis=1)
        else:
            agg_features = pd.DataFrame({'sequence_id': unique_seqs})
        
        # Metadata - single operation
        metadata = df.groupby('sequence_id').agg({
            'subject': 'first',
            'sequence_counter': 'size',
            **({'gesture': 'first'} if is_train and 'gesture' in df.columns else {})
        }).reset_index()
        metadata.rename(columns={'sequence_counter': 'sequence_length'}, inplace=True)
        
        # Vectorized temporal features
        temporal_features = self._extract_temporal_features_fast(df)
        
        # Vectorized ToF features
        tof_features = self._extract_tof_features_fast(df)
        
        # Single merge operation
        result = metadata.merge(agg_features, on='sequence_id', how='left')
        if not temporal_features.empty:
            result = result.merge(temporal_features, on='sequence_id', how='left')
        if not tof_features.empty:
            result = result.merge(tof_features, on='sequence_id', how='left')
        
        # Vectorized fillna
        numeric_cols = result.select_dtypes(include=[np.number]).columns
        result[numeric_cols] = result[numeric_cols].fillna(0)
        
        return result

    def _aggregate_single_sequence(self, df, seq_id, is_train):
        """Optimized single sequence aggregation"""
        # Build feature list
        all_cols = []
        
        # IMU columns
        imu_cols = [col for col in self.sensor_cols if col in df.columns]
        all_cols.extend(imu_cols)
        
        # Temperature columns
        temp_cols = [col for col in self.temp_cols if col in df.columns]
        if temp_cols:
            df['temp_global_mean'] = df[temp_cols].mean(axis=1)
            all_cols.extend(temp_cols + ['temp_global_mean'])
        
        # Missing indicators
        missing_cols = [col for col in df.columns if '_missing_rate' in col or '_key_missing' in col]
        all_cols.extend(missing_cols)
        
        # Initialize result dictionary
        seq_features = {
            'sequence_id': seq_id,
            'subject': df['subject'].iloc[0] if 'subject' in df.columns else 'UNKNOWN',
            'sequence_length': len(df)
        }
        
        if all_cols:
            # Vectorized statistics computation
            data = df[all_cols].values
            
            # Compute all stats at once
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                means = np.nanmean(data, axis=0)
                stds = np.nanstd(data, axis=0)
                maxs = np.nanmax(data, axis=0)
                mins = np.nanmin(data, axis=0)
            
            # Assign all features at once
            for i, col in enumerate(all_cols):
                seq_features[f'{col}_mean'] = means[i]
                seq_features[f'{col}_std'] = stds[i]
                seq_features[f'{col}_max'] = maxs[i]
                seq_features[f'{col}_min'] = mins[i]
                
                if 'acc_' in col:
                    seq_features[f'{col}_range'] = maxs[i] - mins[i]
                    if len(df) > 1:
                        seq_features[f'{col}_temporal_mean'] = np.mean(np.abs(np.diff(data[:, i])))
        
        # Fast ToF features
        tof_features = self._compute_tof_stats_fast(df)
        seq_features.update(tof_features)
        
        if is_train and 'gesture' in df.columns:
            seq_features['gesture'] = df['gesture'].iloc[0]
        
        return pd.DataFrame([seq_features])

    def _extract_temporal_features(self, df):
        """Vectorized temporal features"""
        acc_cols = [col for col in ['acc_x', 'acc_y', 'acc_z'] if col in df.columns]
        if not acc_cols:
            return pd.DataFrame()

        def fast_temporal(group):
            result = {}
            for col in acc_cols:
                values = group[col].values
                if len(values) > 1:
                    result[f'{col}_temporal_mean'] = np.mean(np.abs(np.diff(values)))
                else:
                    result[f'{col}_temporal_mean'] = 0
            return pd.Series(result)
        
        return df.groupby('sequence_id').apply(fast_temporal).reset_index()

    def _extract_tof_features(self, df):
        """Fully vectorized ToF features - much faster"""
        grouped = df.groupby('sequence_id')
        
        # Get sample ToF columns
        tof_sample_cols = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64, 16)]
        tof_sample_cols = [col for col in tof_sample_cols if col in df.columns]
        
        if not tof_sample_cols:
            # No ToF data
            seq_ids = grouped.groups.keys()
            return pd.DataFrame({
                'sequence_id': list(seq_ids),
                'total_working_tof_sensors': 0,
                'imu_only_mode': 1.0,
                'tof_cross_sensor_mean': 254
            })
        
        # Vectorized computation for all sequences at once
        def compute_tof_stats(group):
            tof_data = group[tof_sample_cols].values
            
            # Vectorized valid check
            valid_mask = (tof_data != -1) & (tof_data < 254)
            valid_ratio = valid_mask.mean()
            
            # Estimate working sensors
            working_sensors = min(5, int(valid_ratio * 5 * 1.5))  # Scaling up
            
            # Mean of valid values
            valid_values = tof_data[valid_mask]
            tof_mean = valid_values.mean() if len(valid_values) > 0 else 254
            
            return pd.Series({
                'total_working_tof_sensors': working_sensors,
                'imu_only_mode': 1.0 if valid_ratio < 0.1 else 0.0,
                'tof_cross_sensor_mean': tof_mean
            })
        
        return grouped.apply(compute_tof_stats).reset_index()

    def _compute_tof_stats(self, df):
        """Fast ToF stats for single sequence"""
        # Sample ToF columns
        tof_sample_cols = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64, 16)]
        tof_sample_cols = [col for col in tof_sample_cols if col in df.columns]
        
        if not tof_sample_cols:
            return {
                'total_working_tof_sensors': 0,
                'imu_only_mode': 1.0,
                'tof_cross_sensor_mean': 254
            }
        
        # Vectorized computation
        tof_data = df[tof_sample_cols].values
        valid_mask = (tof_data != -1) & (tof_data < 254)
        valid_ratio = valid_mask.mean()
        
        valid_values = tof_data[valid_mask]
        
        return {
            'total_working_tof_sensors': min(5, int(valid_ratio * 5 * 1.5)),
            'imu_only_mode': 1.0 if valid_ratio < 0.1 else 0.0,
            'tof_cross_sensor_mean': valid_values.mean() if len(valid_values) > 0 else 254
        }

    def demographics(self, seq_df, demo_df):
        """Vectorized demographics merge"""
        if demo_df is None or demo_df.empty or 'subject' not in seq_df.columns:
            # Add default demographic columns
            demographic_cols = ['adult_child', 'age', 'sex', 'handedness', 
                               'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm']
            defaults = [1, 0.25, 0.5, 1, 0.85, 0.275, 0.125]  # Pre-normalized
            
            for col, default in zip(demographic_cols, defaults):
                seq_df[col] = default
            return seq_df
        
        # Merge and normalize in one pass
        result = seq_df.merge(demo_df, on='subject', how='left')
        
        # Vectorized normalization
        norm_cols = {
            'age': 100.0,
            'height_cm': 200.0,
            'shoulder_to_wrist_cm': 200.0,
            'elbow_to_wrist_cm': 200.0
        }
        
        for col, divisor in norm_cols.items():
            if col in result.columns:
                result[col] = result[col].fillna(result[col].median()) / divisor
        
        # Fill other demographics
        for col in ['adult_child', 'sex', 'handedness']:
            if col in result.columns:
                result[col] = result[col].fillna(result[col].mode()[0] if not result[col].mode().empty else 1)
                
        return result
   

class AdvancedFeatureEngineer:
    def __init__(self):
        self.use_reduced_features = False

    def imu_robust_features(self, df):
        new_features = pd.DataFrame(index=df.index)
        
        if all(col in df.columns for col in ['acc_x_mean', 'acc_y_mean', 'acc_z_mean']):
            acc_data = df[['acc_x_mean', 'acc_y_mean', 'acc_z_mean']].values
            acc_magnitude = np.sqrt(np.sum(acc_data**2, axis=1))
            
            new_features['acc_magnitude'] = acc_magnitude
            new_features['acc_magnitude_log'] = np.log1p(acc_magnitude)
            
            if all(col in df.columns for col in ['acc_x_std', 'acc_y_std', 'acc_z_std']):
                acc_std_data = df[['acc_x_std', 'acc_y_std', 'acc_z_std']].values
                acc_jerk = np.mean(acc_std_data, axis=1)
                new_features['acc_jerk'] = acc_jerk
                new_features['movement_smoothness'] = acc_magnitude / (acc_jerk + 1e-8)
                new_features['movement_complexity'] = acc_std_data.std(axis=1)
        
        if all(col in df.columns for col in ['rot_w_mean', 'rot_x_mean', 'rot_y_mean', 'rot_z_mean']):
            # Quaternion magnitude (should be ~1 if normalized)
            quat_data = df[['rot_w_mean', 'rot_x_mean', 'rot_y_mean', 'rot_z_mean']].values
            quat_magnitude = np.sqrt(np.sum(quat_data**2, axis=1))
            new_features['rotation_magnitude'] = quat_magnitude
            
            # Angular velocity approximation
            if all(col in df.columns for col in ['rot_x_std', 'rot_y_std', 'rot_z_std']):
                rot_std = df[['rot_x_std', 'rot_y_std', 'rot_z_std']].values
                new_features['angular_velocity'] = np.sqrt(np.sum(rot_std**2, axis=1))
        
        return new_features

    def sensor_aware_features(self, df):
        new_features = pd.DataFrame(index=df.index)
        
        # Sensor quality score
        if 'total_working_tof_sensors' in df.columns:
            new_features['sensor_quality_score'] = df['total_working_tof_sensors'] / 5.0
            
        # ToF-based proximity features
        tof_cols = [col for col in df.columns if 'tof_' in col and '_mean' in col and '_missing' not in col]
        if tof_cols:
            tof_data = df[tof_cols].values
            # Calculating proximity inverse of distance
            valid_mask = (tof_data > 0) & (tof_data < 254)
            proximity = np.zeros_like(tof_data)
            proximity[valid_mask] = (254 - tof_data[valid_mask]) / 254
            
            new_features['proximity_mean'] = np.mean(proximity, axis=1)
            new_features['proximity_max'] = np.max(proximity, axis=1)
            new_features['proximity_std'] = np.std(proximity, axis=1)
        
        # Temperature variance as activity indicator
        temp_cols = [col for col in df.columns if 'thm_' in col and '_mean' in col]
        if temp_cols:
            temp_data = df[temp_cols].values
            new_features['temp_variance'] = np.var(temp_data, axis=1)
            new_features['temp_range'] = np.ptp(temp_data, axis=1)
        
        return new_features

    def gesture_discrimination_features(self, df):
        new_features = pd.DataFrame(index=df.index)
        
        # Face-level indicators
        if 'acc_z_mean' in df.columns and 'acc_z_max' in df.columns:
            new_features['face_height_indicator'] = df['acc_z_max'] - df['acc_z_mean']
            if 'acc_z_q75' in df.columns:
                new_features['sustained_elevation'] = df['acc_z_q75'] - df['acc_z_q25']
        
        # Movement patterns
        if all(col in df.columns for col in ['acc_x_mean', 'acc_y_mean', 'acc_z_mean']):
            new_features['vertical_dominance'] = np.abs(df['acc_z_mean']) / (
                np.abs(df['acc_x_mean']) + np.abs(df['acc_y_mean']) + 1e-8
            )
            new_features['lateral_movement_ratio'] = np.abs(df['acc_y_mean']) / (
                np.abs(df['acc_z_mean']) + 1e-8
            )
        
        # Repetitive motion indicators
        if 'acc_x_temporal_mean' in df.columns and 'acc_y_temporal_mean' in df.columns:
            new_features['repetitive_motion_score'] = (
                df['acc_x_temporal_mean'] + df['acc_y_temporal_mean']
            ) / 2.0
            
        # Phase-based features for BFRB detection
        phase_cols = [col for col in df.columns if '_ges_' in col or '_pau_' in col or '_tra_' in col]
        if phase_cols:
            # To gesture phase have more activity for BFRBs
            ges_cols = [col for col in phase_cols if '_ges_' in col and '_mean' in col]
            pau_cols = [col for col in phase_cols if '_pau_' in col and '_mean' in col]
            
            if ges_cols and pau_cols:
                ges_activity = df[ges_cols].abs().mean(axis=1)
                pau_activity = df[pau_cols].abs().mean(axis=1)
                new_features['gesture_to_pause_ratio'] = ges_activity / (pau_activity + 1e-8)
        
        # Specific gesture patterns
        if 'acc_z_min' in df.columns:
            new_features['low_hand_indicator'] = -df['acc_z_min']
            
        # Scratch vs pinch differentiation
        if 'acc_x_jerk' in df.columns and 'acc_y_jerk' in df.columns:
            new_features['scratch_pattern'] = df['acc_x_jerk'] + df['acc_y_jerk']
            
        return new_features

    def outliers(self, df, percentiles=(2, 98)):
        df_clean = df.copy()
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        exclude_cols = ['sequence_id', 'subject', 'gesture', 'imu_only_mode', 
                       'total_working_tof_sensors', 'sensor_quality_score']
        process_cols = [col for col in numeric_cols if col not in exclude_cols and '_missing' not in col]
        
        if process_cols:
            lower_bounds = df[process_cols].quantile(percentiles[0] / 100)
            upper_bounds = df[process_cols].quantile(percentiles[1] / 100)
            
            # Special handling for ToF
            tof_cols = [col for col in process_cols if 'tof_' in col]
            for col in tof_cols:
                upper_bounds[col] = 254
                lower_bounds[col] = 0
            
            df_clean[process_cols] = df[process_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)
        
        return df_clean

    def process_features(self, df):
        imu_features = self.imu_robust_features(df)
        sensor_features = self.sensor_aware_features(df)
        gesture_features = self.gesture_discrimination_features(df)
        
        df_enhanced = pd.concat([df, imu_features, sensor_features, gesture_features], axis=1)
        
        # Remove redundant columns
        redundant_cols = []
        
        # Remove highly correlated features
        if 'rot_w_mean' in df_enhanced.columns:
            pass
        
        if redundant_cols:
            cols_to_drop = list(set(redundant_cols))
            df_enhanced = df_enhanced.drop(columns=[col for col in cols_to_drop if col in df_enhanced.columns])
        
        # Fill NaN values
        numeric_cols = df_enhanced.select_dtypes(include=[np.number]).columns
        df_enhanced[numeric_cols] = df_enhanced[numeric_cols].fillna(0)
        
        # Outlier handling
        df_clean = self.outliers(df_enhanced)
        
        return df_clean

class Preprocessor:
    def __init__(self):
        self.label_encoder = LabelEncoder()
        self.scaler = RobustScaler()
        self.feature_columns = None
        self.feature_stats = {}
        self.selected_features = None

    def clean_data(self, df):
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        tof_cols = [col for col in numeric_cols if 'tof_' in col]
        missing_cols = [col for col in numeric_cols if '_missing' in col]
        std_range_cols = [col for col in numeric_cols if '_std' in col or '_range' in col]
        other_cols = [col for col in numeric_cols if col not in tof_cols and col not in std_range_cols and col not in missing_cols]
        
        if tof_cols:
            df[tof_cols] = df[tof_cols].fillna(0)
        if missing_cols:
            df[missing_cols] = df[missing_cols].fillna(0)
        if std_range_cols:
            df[std_range_cols] = df[std_range_cols].fillna(0)
        if other_cols:
            df[other_cols] = df[other_cols].fillna(df[other_cols].median())
        
        return df

    def select_best_features(self, X, y, n_features=120):
        """Increased feature count for better performance"""
        from sklearn.ensemble import RandomForestClassifier
        
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=15,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )
        rf.fit(X, y)
        
        importance = rf.feature_importances_
        indices = np.argsort(importance)[::-1][:n_features]
        
        self.selected_features = indices
        
        return X[:, indices]

    def prepare_features(self, df, is_train=True, use_feature_selection=True, n_features=120):
        exclude_cols = ['sequence_id', 'subject', 'sequence_counter', 'row_id']
        if is_train and 'gesture' in df.columns:
            exclude_cols.append('gesture')
        
        if df.empty:
            return np.zeros((1, n_features)), None, None
        
        if is_train:
            feature_cols = [col for col in df.columns if col not in exclude_cols]
            self.feature_columns = feature_cols
            
            # Store statistics for test time
            self.feature_stats = {}
            for col in feature_cols:
                if col in df.columns:
                    self.feature_stats[col] = {
                        'median': df[col].median(),
                        'mean': df[col].mean(),
                        'std': df[col].std() if df[col].std() > 0 else 1.0
                    }
            
            X = df[feature_cols].astype(float).values
            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
            y = self.label_encoder.fit_transform(df['gesture'])
            
            if use_feature_selection and X.shape[1] > n_features:
                X = self.select_best_features(X, y, n_features)
                if self.selected_features is not None:
                    self.feature_columns = [self.feature_columns[i] for i in self.selected_features]
            
            X = self.scaler.fit_transform(X)
            
            subjects = df['subject'].values if 'subject' in df.columns else None
            return X, y, subjects
        else:
            # Test time
            if self.feature_columns is not None:
                for col in self.feature_columns:
                    if col not in df.columns:
                        if 'tof_' in col:
                            df[col] = 0
                        elif '_missing' in col:
                            df[col] = 0
                        elif '_std' in col or '_range' in col:
                            df[col] = 0
                        elif 'imu_only_mode' in col:
                            df[col] = 1.0
                        elif col in self.feature_stats:
                            df[col] = self.feature_stats[col].get('median', 0)
                        else:
                            df[col] = 0
                
                feature_cols = self.feature_columns
            else:
                feature_cols = [col for col in df.columns if col not in exclude_cols]
        
            X = df[feature_cols].astype(float).values
            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
            
            if self.selected_features is not None and self.feature_columns is not None:
                if X.shape[1] > len(self.selected_features):
                    X = X[:, self.selected_features]
            
            X = self.scaler.transform(X)
            
            subjects = df['subject'].values if 'subject' in df.columns else None
            return X, subjects

class CNNLSTMModel:
    def __init__(self, input_dim, num_classes):
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.model = None
        self.history = None

    def build_hybrid_model(self):
        """Improved model architecture with attention"""
        inputs = Input(shape=(self.input_dim,))

        # First pathway - main processing
        x = layers.Dense(256, activation='gelu')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        
        # Attention mechanism
        attention = layers.Dense(256, activation='sigmoid')(x)
        x = layers.Multiply()([x, attention])

        x = layers.Dense(128, activation='gelu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.25)(x)

        x = layers.Dense(64, activation='gelu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.2)(x)

        # Residual connection
        res = layers.Dense(64, activation='gelu')(inputs)
        x = layers.Add()([x, res])

        x = layers.Dense(32, activation='gelu')(x)
        x = layers.Dropout(0.15)(x)

        outputs = layers.Dense(self.num_classes, activation='softmax')(x)

        self.model = Model(inputs=inputs, outputs=outputs)
        return self

    def smart_augmentation(self, X_train, y_train):
        """Enhanced augmentation with mixup"""
        from collections import Counter
        
        class_counts = Counter(y_train)
        target_count = int(np.median(list(class_counts.values())) * 1.8)
        
        # More aggressive for problem classes
        problem_multipliers = {
            13: 3.0, 3: 2.5, 9: 2.5, 10: 2.5,
            11: 2.0, 1: 1.8, 4: 1.8, 0: 1.5, 6: 1.5, 7: 1.5
        }
        
        aug_sizes = []
        aug_classes = []
        for class_id, count in class_counts.items():
            multiplier = problem_multipliers.get(class_id, 1.5)
            target = int(min(target_count * multiplier, count * 3))
            n_aug = target - count
            if n_aug > 0:
                aug_sizes.append(n_aug)
                aug_classes.append(class_id)
        
        if not aug_sizes:
            return X_train, y_train
        
        total_aug = sum(aug_sizes)
        aug_X = np.zeros((total_aug, X_train.shape[1]))
        aug_y = np.zeros(total_aug, dtype=int)
        
        current_idx = 0
        for class_id, n_aug in zip(aug_classes, aug_sizes):
            class_mask = y_train == class_id
            class_samples = X_train[class_mask]
            
            if len(class_samples) == 0:
                continue
            
            # Select samples for augmentation
            sample_indices = np.random.choice(len(class_samples), n_aug, replace=True)
            augmented = class_samples[sample_indices].copy()
            
            # Apply different augmentation strategies
            aug_type = np.random.rand(n_aug)
            
            # Gaussian noise  with 40%
            noise_mask = aug_type < 0.4
            if noise_mask.sum() > 0:
                noise_level = 0.03 if class_id in problem_multipliers else 0.02
                augmented[noise_mask] += np.random.normal(0, noise_level, 
                                                         (noise_mask.sum(), X_train.shape[1]))
            
            # Scaling 30%
            scale_mask = (aug_type >= 0.4) & (aug_type < 0.7)
            if scale_mask.sum() > 0:
                scale_factors = np.random.uniform(0.85, 1.15, (scale_mask.sum(), 1))
                augmented[scale_mask] *= scale_factors
            
            # Mixup 30%
            mixup_mask = aug_type >= 0.7
            if mixup_mask.sum() > 0:
                # Mixing with random samples from same class
                mix_indices = np.random.choice(len(class_samples), mixup_mask.sum(), replace=True)
                alpha = np.random.beta(0.2, 0.2, (mixup_mask.sum(), 1))
                augmented[mixup_mask] = (alpha * augmented[mixup_mask] + 
                                        (1 - alpha) * class_samples[mix_indices])
            
            aug_X[current_idx:current_idx+n_aug] = augmented
            aug_y[current_idx:current_idx+n_aug] = class_id
            current_idx += n_aug
        
        X_combined = np.vstack([X_train, aug_X[:current_idx]])
        y_combined = np.hstack([y_train, aug_y[:current_idx]])
        
        # Shuffle
        shuffle_idx = np.random.permutation(len(X_combined))
        X_combined = X_combined[shuffle_idx]
        y_combined = y_combined[shuffle_idx]
        
        return X_combined, y_combined

    def train(self, X_train, y_train, X_val=None, y_val=None, epochs=60):
        X_train, y_train = self.smart_augmentation(X_train, y_train)
        
        # Class weights for imbalanced data
        from sklearn.utils.class_weight import compute_class_weight
        classes = np.unique(y_train)
        class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
        class_weight_dict = dict(zip(classes, class_weights))
        
        # Initial learning rate
        initial_lr = 0.002
        
        # Cosine Annealing with Warm Restarts
        def cosine_annealing_schedule(epoch, lr):
            """Cosine annealing with warm restarts every 20 epochs"""
            epochs_per_restart = 20
            epoch_in_cycle = epoch % epochs_per_restart
            cosine_decay = 0.5 * (1 + np.cos(np.pi * epoch_in_cycle / epochs_per_restart))
            decayed_lr = initial_lr * cosine_decay
            
            # Reduce initial LR after each restart
            num_restarts = epoch // epochs_per_restart
            decayed_lr = decayed_lr * (0.9 ** num_restarts)  # Reduce by 10% each restart
            
            return decayed_lr
    
        # Creating the scheduler callback
        lr_scheduler = tf.keras.callbacks.LearningRateScheduler(
            cosine_annealing_schedule,
            verbose=0
        )
        
        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=initial_lr),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        callbacks = [
            # Cosine annealing scheduler
            lr_scheduler,
            
            # ReduceLROnPlateau as backup
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='loss',
                factor=0.5,
                patience=8,
                min_lr=1e-7,
                verbose=0
            ),
        
            # Early stopping
            tf.keras.callbacks.EarlyStopping(
                monitor='loss', 
                patience=15,
                restore_best_weights=True,
                verbose=0
            )
        ]
        
        self.history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=32,
            verbose=1,
            callbacks=callbacks,
            class_weight=class_weight_dict,
            shuffle=True
        )
        
        return self

    def predict(self, X):
        return np.argmax(self.model.predict(X, verbose=0, batch_size=64), axis=1)

    def predict_proba(self, X):
        return self.model.predict(X, verbose=0, batch_size=64)


# Global variables initialization
print("Initializing global variables...")
missing_handler = None
extractor = None
engineer = None
preprocessor = None
nn_model = None
nn_model_imu_only = None
imu_feature_indices = None

# Training section
print("Starting training process...")

try:
    print("Step 1: Loading data...")
    loader = DataLoader()
    loader.load_data()
    print(f"Loaded train data: {loader.train_df.shape}")
    
    print("Step 2: Handling missing data...")
    missing_handler = MissingDataHandler()
    train_processed = missing_handler.process(loader.train_df.copy(), is_train=True)
    print(f"Processed data shape: {train_processed.shape}")
    
    print("Step 3: Extracting features...")
    extractor = FeatureExtractor()
    train_seq = extractor.aggregate_sequence(train_processed, is_train=True)
    print(f"Aggregated sequences: {train_seq.shape}")
    
    train_seq = extractor.demographics(train_seq, loader.train_demo)
    print(f"Added demographics: {train_seq.shape}")
    
    print("Step 4: Engineering features...")
    engineer = AdvancedFeatureEngineer()
    train_seq_engineered = engineer.process_features(train_seq)
    print(f"Engineered features: {train_seq_engineered.shape}")
    
    print("Step 5: Preprocessing...")
    preprocessor = Preprocessor()
    train_seq_clean = preprocessor.clean_data(train_seq_engineered.copy())
    
    X_train_full, y_train_full, subjects = preprocessor.prepare_features(
        train_seq_clean, is_train=True, use_feature_selection=True, n_features=120
    )
    
    print(f"Final training shape: {X_train_full.shape}")
    print(f"Number of classes: {len(np.unique(y_train_full))}")
    print(f"Class distribution: {np.bincount(y_train_full)}")
    
    print("Step 6: Training main model...")
    nn_model = CNNLSTMModel(input_dim=X_train_full.shape[1], num_classes=len(np.unique(y_train_full)))
    nn_model.build_hybrid_model()
    nn_model.train(X_train_full, y_train_full, epochs=50)
    
    print("Step 7: Creating IMU-only feature set...")
    # A version with ONLY IMU features
    imu_only_cols = []
    for i, col in enumerate(preprocessor.feature_columns):
        # Keep only IMU and demographic features
        if any(x in col.lower() for x in ['acc_', 'rot_', 'age', 'sex', 'height', 
                                            'adult', 'handed', 'wrist', 'elbow',
                                            'sequence_length', 'temporal', 'jerk',
                                            'magnitude', 'smoothness', 'movement',
                                            'angular', 'vertical', 'lateral',
                                            'elevation', 'dominance', 'ratio']):
            imu_only_cols.append(i)

    imu_feature_indices = imu_only_cols
    print(f"Using {len(imu_only_cols)} IMU-only features out of {len(preprocessor.feature_columns)} total")
    
    # Extracting only IMU features
    X_imu = X_train_full[:, imu_only_cols]
    
    print("Step 8: Training IMU-only model...")
    # A smaller model for IMU-only features
    nn_model_imu_only = CNNLSTMModel(
        input_dim=len(imu_only_cols),
        num_classes=len(np.unique(y_train_full))
    )
    nn_model_imu_only.build_hybrid_model()
    
    # Training with higher learning rate and more epochs
    nn_model_imu_only.model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),  # Higher LR
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    nn_model_imu_only.train(X_imu, y_train_full, epochs=80)  # More epochs
    
    print("Training completed successfully!")


except Exception as e:
    print(f"Training error: {e}")
    import traceback
    traceback.print_exc()

    if missing_handler is None:
        missing_handler = MissingDataHandler()
    if extractor is None:
        extractor = FeatureExtractor()
    if engineer is None:
        engineer = AdvancedFeatureEngineer()
    if preprocessor is None:
        preprocessor = Preprocessor()
        preprocessor.label_encoder.fit(GESTURE_NAMES)
    if nn_model is None:
        nn_model = CNNLSTMModel(input_dim=120, num_classes=18)
        nn_model.build_hybrid_model()
    if nn_model_imu_only is None:
        nn_model_imu_only = CNNLSTMModel(input_dim=120, num_classes=18)
        nn_model_imu_only.build_hybrid_model()


# def evaluate_model(model, X_test, y_test, preprocessor, class_names=None):
#     """
#     Evaluate model performance with metrics and visualizations
    
#     Args:
#         model: Trained model (nn_model)
#         X_test: Test features
#         y_test: True labels
#         preprocessor: Preprocessor object with label_encoder
#         class_names: Optional list of class names
#     """
#     import matplotlib.pyplot as plt
#     from sklearn.metrics import confusion_matrix, classification_report, f1_score
#     import seaborn as sns
    
#     # Get predictions
#     y_pred = model.predict(X_test)
#     y_pred_proba = model.predict_proba(X_test)
    
#     # Get class names
#     if class_names is None:
#         class_names = preprocessor.label_encoder.classes_
    
#     # Calculate metrics
#     print("=" * 60)
#     print("MODEL EVALUATION METRICS")
#     print("=" * 60)
    
#     # Overall accuracy
#     accuracy = np.mean(y_pred == y_test)
#     print(f"\nOverall Accuracy: {accuracy:.4f}")
    
#     # F1 Scores
#     f1_macro = f1_score(y_test, y_pred, average='macro')
#     f1_weighted = f1_score(y_test, y_pred, average='weighted')
#     print(f"F1 Score (Macro): {f1_macro:.4f}")
#     print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    
    # # Binary classification for BFRB vs Non-BFRB
    # bfrb_indices = {0, 1, 3, 4, 6, 7, 9, 10, 13}
    # y_test_binary = np.array([1 if y in bfrb_indices else 0 for y in y_test])
    # y_pred_binary = np.array([1 if y in bfrb_indices else 0 for y in y_pred])
    # binary_f1 = f1_score(y_test_binary, y_pred_binary)
    # print(f"Binary F1 (BFRB vs Non-BFRB): {binary_f1:.4f}")
    
    # # Competition metric (average of binary and macro F1)
    # competition_score = (binary_f1 + f1_macro) / 2
    # print(f"\n>>> COMPETITION SCORE (Avg of Binary & Macro F1): {competition_score:.4f} <<<")
    
    # # Per-class metrics
    # print("\n" + "=" * 60)
    # print("PER-CLASS METRICS")
    # print("=" * 60)
    # print(classification_report(y_test, y_pred, target_names=[str(cn)[:30] for cn in class_names]))
    
    # # Create visualizations
    # fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # # 1. Confusion Matrix
    # cm = confusion_matrix(y_test, y_pred)
    # sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
    # axes[0, 0].set_title('Confusion Matrix')
    # axes[0, 0].set_xlabel('Predicted')
    # axes[0, 0].set_ylabel('Actual')
    
    # # 2. Normalized Confusion Matrix
    # cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    # sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0, 1])
    # axes[0, 1].set_title('Normalized Confusion Matrix')
    # axes[0, 1].set_xlabel('Predicted')
    # axes[0, 1].set_ylabel('Actual')
    
    # # 3. Prediction Confidence Distribution
    # max_probs = np.max(y_pred_proba, axis=1)
    # axes[1, 0].hist(max_probs, bins=50, edgecolor='black')
    # axes[1, 0].set_title('Prediction Confidence Distribution')
    # axes[1, 0].set_xlabel('Max Probability')
    # axes[1, 0].set_ylabel('Count')
    # axes[1, 0].axvline(x=max_probs.mean(), color='red', linestyle='--', label=f'Mean: {max_probs.mean():.3f}')
    # axes[1, 0].legend()
    
    # # 4. Residuals (Error Analysis)
    # errors = (y_pred != y_test).astype(int)
    # error_by_class = []
    # for i in range(len(class_names)):
    #     mask = y_test == i
    #     if mask.sum() > 0:
    #         error_rate = errors[mask].mean()
    #         error_by_class.append(error_rate)
    #     else:
    #         error_by_class.append(0)
    
    # axes[1, 1].bar(range(len(class_names)), error_by_class)
    # axes[1, 1].set_title('Error Rate by Class')
    # axes[1, 1].set_xlabel('Class Index')
    # axes[1, 1].set_ylabel('Error Rate')
    # axes[1, 1].set_xticks(range(len(class_names)))
    # axes[1, 1].set_xticklabels(range(len(class_names)), rotation=45)
    
    # plt.tight_layout()
    # plt.show()
    
    # # Most confused pairs
    # print("\n" + "=" * 60)
    # print("TOP 5 MOST CONFUSED PAIRS")
    # print("=" * 60)
    
    # # Find most confused pairs
    # confused_pairs = []
    # for i in range(len(class_names)):
    #     for j in range(len(class_names)):
    #         if i != j and cm[i, j] > 0:
    #             confused_pairs.append((cm[i, j], i, j))
    
    # confused_pairs.sort(reverse=True)
    # for count, actual, predicted in confused_pairs[:5]:
    #     actual_name = class_names[actual][:30]
    #     predicted_name = class_names[predicted][:30]
    #     print(f"{count:3d} times: {actual_name} -> {predicted_name}")
    
    # return {
    #     'accuracy': accuracy,
    #     'f1_macro': f1_macro,
    #     'f1_weighted': f1_weighted,
    #     'binary_f1': binary_f1,
    #     'competition_score': competition_score,
    #     'confusion_matrix': cm,
    #     'max_probs': max_probs
    # }

# # Simple validation split function
# def create_validation_split(X, y, subjects=None, val_size=0.2, random_state=42):
#     """Create validation split, preferably by subject"""
#     if subjects is not None:
#         # Group by subject to prevent leakage
#         unique_subjects = np.unique(subjects)
#         np.random.seed(random_state)
#         np.random.shuffle(unique_subjects)
        
#         n_val = int(len(unique_subjects) * val_size)
#         val_subjects = unique_subjects[:n_val]
        
#         val_mask = np.isin(subjects, val_subjects)
#         train_mask = ~val_mask
        
#         return X[train_mask], X[val_mask], y[train_mask], y[val_mask]
#     else:
#         # Regular split
#         from sklearn.model_selection import train_test_split
#         return train_test_split(X, y, test_size=val_size, random_state=random_state, stratify=y)


# # FORCE THE EVALUATION TO RUN
# print("\n" + "="*60)
# print("CHECKING IF EVALUATION CAN RUN...")
# print("="*60)

# # Debug prints to see what's available
# print(f"nn_model exists: {'nn_model' in globals() and nn_model is not None}")
# print(f"nn_model_imu_only exists: {'nn_model_imu_only' in globals() and nn_model_imu_only is not None}")
# print(f"X_train_full exists: {'X_train_full' in globals()}")
# print(f"y_train_full exists: {'y_train_full' in globals()}")
# print(f"preprocessor exists: {'preprocessor' in globals() and preprocessor is not None}")


            #     if any(x in col.lower() for x in ['acc_', 'rot_', 'age', 'sex', 'height', 
            #                                        'adult', 'handed', 'wrist', 'elbow',
            #                                        'sequence_length', 'temporal', 'jerk',
            #                                        'magnitude', 'smoothness', 'movement',
            #                                        'angular', 'vertical', 'lateral',
            #                                        'elevation', 'dominance', 'ratio']):
            #         imu_only_cols.append(i)
            
            # print(f"Using {len(imu_only_cols)} IMU features for evaluation")
            
            # # Extract IMU features using saved indices
            # X_val_imu = X_val_split[:, imu_feature_indices]
            # print(f"IMU validation shape: {X_val_imu.shape}")
            
            # # Create a wrapper model for evaluation that handles the feature extraction
            # class IMUModelWrapper:
            #     def __init__(self, model):
            #         self.model = model
                    
            #     def predict(self, X):
            #         # X is already the correct IMU features
            #         return self.model.predict(X)
                
            #     def predict_proba(self, X):
            #         # X is already the correct IMU features
            #         return self.model.predict_proba(X)
            
            # imu_wrapper = IMUModelWrapper(nn_model_imu_only)
            
#             # Evaluate with IMU features only
#             metrics_imu = evaluate_model(
#                 imu_wrapper,
#                 X_val_imu,  # Use IMU features only
#                 y_val_split,
#                 preprocessor,
#                 class_names=gesture_classes
#             )
            
#             print("\n" + "="*60)
#             print(f"Main Model Competition Score: {metrics_main['competition_score']:.4f}")
#             print(f"IMU Model Competition Score: {metrics_imu['competition_score']:.4f}")
#             print("="*60)
#         else:
#             print("IMU model not available for evaluation")
        
#     except Exception as e:
#         print(f"Evaluation error: {e}")
#         import traceback
#         traceback.print_exc()
# else:
#     print("Cannot run evaluation - missing required variables")

#     print(f"nn_model: {nn_model if 'nn_model' in globals() else 'NOT DEFINED'}")
#     print(f"X_train_full shape: {X_train_full.shape if 'X_train_full' in globals() else 'NOT DEFINED'}")
    

def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    """Prediction with better sensor-adaptive processing"""
    global missing_handler, extractor, engineer, preprocessor, nn_model, nn_model_imu_only, gesture_classes, imu_feature_indices
    
    try:
        # Convert to pandas
        sequence_df = sequence.to_pandas()
        demographics_df = demographics.to_pandas() if demographics is not None and not demographics.is_empty() else pd.DataFrame()
        
        if 'sequence_id' not in sequence_df.columns:
            sequence_df['sequence_id'] = 1
        
        # Process the sequence
        sequence_processed = missing_handler.process(sequence_df.copy(), is_train=False)
        seq_features = extractor.aggregate_sequence(sequence_processed, is_train=False)
        
        if not demographics_df.empty:
            seq_features = extractor.demographics(seq_features, demographics_df)
        else:
            seq_features = extractor.demographics(seq_features, None)
        
        seq_engineered = engineer.process_features(seq_features)
        seq_clean = preprocessor.clean_data(seq_engineered.copy())
        X_test, _ = preprocessor.prepare_features(seq_clean, is_train=False)
        
        # Assess sensor quality
        sensor_quality = 1.0
        tof_missing_rate = 0.0
        
        # Check IMU-only mode
        if 'imu_only_mode' in seq_features.columns:
            imu_only = seq_features['imu_only_mode'].iloc[0]
        else:
            imu_only = 0
            
        # Check ToF missing rate
        if 'tof_missing_rate' in seq_features.columns:
            tof_missing_rate = seq_features['tof_missing_rate'].iloc[0]
        
        # Calculate overall sensor quality
        tof_cols = [col for col in sequence_df.columns if 'tof_' in col]
        if tof_cols:
            tof_data = sequence_df[tof_cols].values
            # Check how many valid readings we have
            valid_ratio = np.mean((tof_data > 0) & (tof_data < 254))
            
            if valid_ratio < 0.1:
                sensor_quality = 0.2
            elif valid_ratio < 0.3:
                sensor_quality = 0.4
            elif valid_ratio < 0.5:
                sensor_quality = 0.6
            else:
                sensor_quality = 0.8 + (valid_ratio * 0.2)
        else:
            sensor_quality = 0.3
        
        # Get predictions from both models
        if nn_model_imu_only is not None:
            # Main model prediction with all features
            pred_main = nn_model.predict_proba(X_test)[0]
            
            # IMU only IMU features
            if 'imu_feature_indices' in globals() and imu_feature_indices is not None:
                X_test_imu = X_test[:, imu_feature_indices]
            else:
                # Fallback to extracting them again
                imu_only_cols = []
                for i, col in enumerate(preprocessor.feature_columns):
                    if any(x in col.lower() for x in ['acc_', 'rot_', 'age', 'sex', 'height', 
                                                       'adult', 'handed', 'wrist', 'elbow',
                                                       'sequence_length', 'temporal', 'jerk',
                                                       'magnitude', 'smoothness', 'movement',
                                                       'angular', 'vertical', 'lateral',
                                                       'elevation', 'dominance', 'ratio']):
                        imu_only_cols.append(i)
                X_test_imu = X_test[:, imu_only_cols] if imu_only_cols else X_test
            
            # Get IMU model prediction
            pred_imu = nn_model_imu_only.predict_proba(X_test_imu)[0]
            
            # Adaptive weighting based on sensor quality
            if sensor_quality < 0.3:
                # Very poor sensors
                main_weight = 0.3
                imu_weight = 0.7
            elif sensor_quality < 0.5:
                # Poor sensors - balanced
                main_weight = 0.5
                imu_weight = 0.5
            elif sensor_quality < 0.7:
                # Moderate sensors
                main_weight = 0.7
                imu_weight = 0.3
            else:
                # Good sensors
                main_weight = 0.85
                imu_weight = 0.15
            
            # Weighted ensemble
            pred_proba = main_weight * pred_main + imu_weight * pred_imu
            
            # Test-time augmentation for low-quality sensors
            if sensor_quality < 0.5:
                tta_preds = []
                for _ in range(3):
                    # Augment full features
                    X_aug = X_test + np.random.normal(0, 0.015, X_test.shape)
                    pred_aug_main = nn_model.predict_proba(X_aug)[0]
                    
                    # Augment IMU features
                    X_aug_imu = X_test_imu + np.random.normal(0, 0.015, X_test_imu.shape)
                    pred_aug_imu = nn_model_imu_only.predict_proba(X_aug_imu)[0]
                    
                    pred_aug = main_weight * pred_aug_main + imu_weight * pred_aug_imu
                    tta_preds.append(pred_aug)
                
                # Combine original and augmented predictions
                pred_proba = 0.5 * pred_proba + 0.5 * np.mean(tta_preds, axis=0)
        else:
            # Only main model available
            pred_proba = nn_model.predict_proba(X_test)[0]
        
        # Temperature scaling for calibration
        temperature = 1.2 if sensor_quality < 0.5 else 1.0
        if temperature != 1.0:
            pred_proba = np.power(pred_proba, 1/temperature)
            pred_proba = pred_proba / pred_proba.sum()
        
        # Get predicted class
        predicted_idx = np.argmax(pred_proba)
        
        # Post-processing for common confusions
        top3_indices = np.argsort(pred_proba)[-3:][::-1]
        conf_best = pred_proba[predicted_idx]
        
        # Handle specific confusion pairs
        if len(top3_indices) > 1:
            second_idx = top3_indices[1]
            conf_second = pred_proba[second_idx]
            conf_diff = conf_best - conf_second
            
            # Neck pinch vs scratch confusion
            if (predicted_idx == 9 and second_idx == 10) or (predicted_idx == 10 and second_idx == 9):
                if conf_diff < 0.2:
                    # When uncertain, prefer scratch
                    predicted_idx = 10 if pred_proba[10] > pred_proba[9] * 0.85 else 9
            
            # Scratch knee/leg skin confusion with pinch
            elif predicted_idx == 13 and second_idx == 11:
                if conf_best < 0.4 and sensor_quality < 0.5:
                    predicted_idx = 11
            elif predicted_idx == 11 and second_idx == 13:
                if conf_best < 0.35:
                    predicted_idx = 13
            
            # Forehead gestures confusion
            elif (predicted_idx == 6 and second_idx == 7) or (predicted_idx == 7 and second_idx == 6):
                if conf_diff < 0.15:
                    # Prefer scratch when uncertain
                    predicted_idx = 7 if pred_proba[7] > pred_proba[6] * 0.9 else 6
            
            # Hair pulling confusions (based on your confusion matrix)
            elif predicted_idx in [0, 3, 4] and second_idx in [0, 3, 4]:
                if conf_diff < 0.2:
                    # Eyebrow vs Eyelash confusion is common
                    if predicted_idx == 3 and second_idx == 4:
                        # Keep eyebrow if confidence is decent
                        if conf_best < 0.3:
                            predicted_idx = 4  # Switch to eyelash
                    elif predicted_idx == 4 and second_idx == 3:
                        # Keep eyelash if confidence is decent
                        if conf_best < 0.3:
                            predicted_idx = 3  # Switch to eyebrow

                    if 'proximity_mean' in seq_features.columns:
                        prox = seq_features['proximity_mean'].iloc[0]
                        if prox > 0.6:  # Close to face
                            predicted_idx = 3  # Eyebrow
                        elif prox > 0.4:
                            predicted_idx = 4  # Eyelash
                        else:
                            predicted_idx = 0  # Above ear
        
        # Low confidence fallback
        if conf_best < 0.2:
            # Fall back to common gestures
            common_gestures = [14, 0, 10, 7, 3, 2]  # Text on phone, Above ear, Neck scratch, etc.
            for gesture_idx in common_gestures:
                if gesture_idx in top3_indices:
                    predicted_idx = gesture_idx
                    break
        
        # Special handling for IMU-only mode
        if sensor_quality < 0.3 and conf_best < 0.4:
            # For very poor sensors with low confidence
            bfrb_indices = {0, 1, 3, 4, 6, 7, 9, 10, 13}
            bfrb_probs = sum(pred_proba[i] for i in bfrb_indices)
            
            if bfrb_probs > 0.5:
                # Predict most likely BFRB
                bfrb_scores = [(i, pred_proba[i]) for i in bfrb_indices]
                predicted_idx = max(bfrb_scores, key=lambda x: x[1])[0]
            else:
                # Predict most likely non-BFRB
                non_bfrb_indices = {2, 5, 8, 11, 12, 14, 15, 16, 17}
                non_bfrb_scores = [(i, pred_proba[i]) for i in non_bfrb_indices]
                predicted_idx = max(non_bfrb_scores, key=lambda x: x[1])[0]
        
        return gesture_classes[predicted_idx]
        
    except Exception as e:
        print(f"Prediction error: {e}")
        # Return most common gesture as fallback
        return "Text on phone"
        

# Set up inference server
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        data_paths=(
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
        )
    )

2025-08-07 02:55:01.912585: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754535302.113646      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754535302.171806      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Initializing global variables...
Starting training process...
Step 1: Loading data...
Loaded train data: (574945, 341)
Step 2: Handling missing data...
Processed data shape: (574945, 351)
Step 3: Extracting features...
Aggregated sequences: (8151, 105)
Added demographics: (8151, 112)
Step 4: Engineering features...
Engineered features: (8151, 130)
Step 5: Preprocessing...
Final training shape: (8151, 120)
Number of classes: 18
Class distribution: [638 637 161 638 640 161 640 640 161 640 640 161 477 161 640 478 477 161]
Step 6: Training main model...


I0000 00:00:1754535394.107008      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/50


2025-08-07 02:56:35.654037: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


706/706 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.2753 - loss: 2.0207 - learning_rate: 0.0020
Epoch 2/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.4851 - loss: 1.1712 - learning_rate: 0.0020
Epoch 3/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5441 - loss: 1.0080 - learning_rate: 0.0020
Epoch 4/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.5894 - loss: 0.9042 - learning_rate: 0.0019
Epoch 5/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6147 - loss: 0.8385 - learning_rate: 0.0018
Epoch 6/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6370 - loss: 0.7717 - learning_rate: 0.0017
Epoch 7/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6574 - loss: 0.7274 - learning_rate: 0.0016
Epoch 8/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6825 - loss: 0.6745 - learning_rate: 0.0015
Epoch 9/50
706/706 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6987 - loss: 0.6281 - learning_rate: 0.0013
Epoch 10/50
706/706 